# 2_Modelo_Preditivo

Objetivo: prever **risco de defasagem no ano seguinte** com foco em alta sensibilidade (Recall).

## 1) Setup

In [1]:
import re
import unicodedata
from pathlib import Path
import warnings
import pickle

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    classification_report,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier

warnings.filterwarnings("ignore")
px.defaults.template = "plotly_white"


def detect_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "app.py").exists():
            return candidate
    return current


PROJECT_ROOT = detect_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "pede_consolidado_2022_2024.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 5
TARGET_IAN_THRESHOLD = 5.0
TARGET_IDA_THRESHOLD = 6.0
RECALL_OBJECTIVE = 0.80
THRESHOLD_GRID = np.round(np.linspace(0.10, 0.90, 33), 3)


## 2) Funcoes de carga e harmonizacao

In [2]:
def normalize_column_name(col_name: str) -> str:
    normalized = unicodedata.normalize("NFKD", str(col_name)).encode("ascii", "ignore").decode("ascii")
    normalized = re.sub(r"[^0-9a-zA-Z]+", "_", normalized).strip("_").lower()
    return normalized


def clean_empty_strings(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip()
    return cleaned.replace({"": np.nan, "nan": np.nan, "None": np.nan, "<NA>": np.nan})


def coerce_numeric_series(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "<NA>": np.nan, "INCLUIR": np.nan, "incluir": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def normalize_age_series(series: pd.Series) -> pd.Series:
    numeric_age = coerce_numeric_series(series)
    date_values = pd.to_datetime(series, errors="coerce")

    age_from_date = np.where(
        date_values.notna() & (date_values.dt.year == 1900) & (date_values.dt.month == 1),
        date_values.dt.day,
        np.nan,
    )

    result = pd.Series(numeric_age, index=series.index)
    mask = result.isna() & ~pd.isna(age_from_date)
    result.loc[mask] = age_from_date[mask]
    result = result.where(result.between(6, 30))
    return result.round()


def coalesce_columns(df: pd.DataFrame, new_col: str, candidates: list[str], numeric: bool = False) -> None:
    result = None
    for col in candidates:
        if col not in df.columns:
            continue
        values = coerce_numeric_series(df[col]) if numeric else clean_empty_strings(df[col])
        result = values if result is None else result.where(result.notna(), values)
    if result is None:
        result = pd.Series(np.nan, index=df.index)
    df[new_col] = result


def standardize_gender(value: str) -> str:
    if pd.isna(value):
        return np.nan
    value_clean = str(value).strip().lower()
    mapping = {"menino": "Masculino", "masculino": "Masculino", "menina": "Feminino", "feminino": "Feminino"}
    return mapping.get(value_clean, str(value).strip())


def standardize_stone(value: str) -> str:
    if pd.isna(value):
        return np.nan
    value_norm = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii")
    value_norm = value_norm.strip().lower()
    mapping = {
        "quartzo": "Quartzo",
        "agata": "Agata",
        "ametista": "Ametista",
        "topazio": "Topazio",
        "incluir": np.nan,
    }
    return mapping.get(value_norm, str(value).strip())


def extract_phase_code(value: str) -> str:
    if pd.isna(value):
        return np.nan
    raw = str(value).strip().upper()
    if raw in {"", "NAN", "NONE"}:
        return np.nan
    if "ALFA" in raw:
        return "ALFA"
    match = re.search(r"FASE\s*([0-9]+)", raw)
    if match:
        return f"FASE {int(match.group(1))}"
    if re.fullmatch(r"[0-9]+", raw):
        phase_num = int(raw)
        return "ALFA" if phase_num == 0 else f"FASE {phase_num}"
    return np.nan


def load_and_prepare_data(data_path: Path) -> pd.DataFrame:
    xls = pd.ExcelFile(data_path)
    all_frames = []

    for sheet in xls.sheet_names:
        year_match = re.search(r"20\d{2}", sheet)
        if not year_match:
            continue
        year = int(year_match.group())

        raw = xls.parse(sheet)
        raw = raw.dropna(how="all").copy()
        raw.columns = [normalize_column_name(c) for c in raw.columns]

        if "ra" not in raw.columns:
            continue
        raw["ra"] = clean_empty_strings(raw["ra"])
        raw = raw[~raw["ra"].str.upper().isin(["RA-1", "NAN"])]
        raw = raw[raw["ra"].notna()]
        raw["ano_referencia"] = year

        coalesce_columns(raw, "idade", ["idade", "idade_22"], numeric=False)
        coalesce_columns(raw, "instituicao_ensino", ["instituicao_de_ensino", "escola"], numeric=False)
        coalesce_columns(raw, "fase_ideal", ["fase_ideal", "fase_ideal_"], numeric=False)
        coalesce_columns(raw, "defasagem", ["defasagem", "defas"], numeric=True)
        coalesce_columns(raw, "nota_matematica", ["mat", "matem"], numeric=True)
        coalesce_columns(raw, "nota_portugues", ["por", "portug"], numeric=True)
        coalesce_columns(raw, "nota_ingles", ["ing", "ingles"], numeric=True)
        coalesce_columns(raw, "inde_ano", ["inde_2024", "inde_2023", "inde_23", "inde_22"], numeric=True)
        coalesce_columns(raw, "pedra_ano", ["pedra_2024", "pedra_2023", "pedra_23", "pedra_22"], numeric=False)
        coalesce_columns(raw, "ativo_inativo", ["ativo_inativo", "ativo_inativo_1"], numeric=False)

        for col in ["fase", "turma", "genero", "ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv", "inde_22", "inde_23"]:
            if col not in raw.columns:
                raw[col] = np.nan

        phase_from_fase = raw["fase"].apply(extract_phase_code)
        phase_from_ideal = raw["fase_ideal"].apply(extract_phase_code)
        raw["fase_programa"] = phase_from_fase.where(phase_from_fase.notna(), phase_from_ideal)

        raw["idade"] = normalize_age_series(raw["idade"])

        for num_col in [
            "defasagem",
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]:
            raw[num_col] = coerce_numeric_series(raw[num_col])

        for score_col in [
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]:
            raw[score_col] = raw[score_col].clip(lower=0, upper=10)

        raw["genero"] = raw["genero"].apply(standardize_gender)
        raw["pedra_ano"] = raw["pedra_ano"].apply(standardize_stone)

        keep_cols = [
            "ra",
            "ano_referencia",
            "idade",
            "genero",
            "instituicao_ensino",
            "fase_programa",
            "turma",
            "pedra_ano",
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "defasagem",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
            "ativo_inativo",
        ]
        for col in keep_cols:
            if col not in raw.columns:
                raw[col] = np.nan

        all_frames.append(raw[keep_cols].copy())

    if not all_frames:
        raise ValueError("Nenhuma aba valida foi encontrada para consolidacao.")

    df = pd.concat(all_frames, ignore_index=True)
    df = df.drop_duplicates(subset=["ra", "ano_referencia"], keep="first")
    df = df.sort_values(["ra", "ano_referencia"]).reset_index(drop=True)

    for cat_col in ["genero", "instituicao_ensino", "fase_programa", "turma", "pedra_ano", "ativo_inativo"]:
        df[cat_col] = clean_empty_strings(df[cat_col])

    return df


def load_consolidated_or_raw() -> pd.DataFrame:
    if PROCESSED_PATH.exists():
        df = pd.read_csv(PROCESSED_PATH)
        if "idade" in df.columns:
            df["idade"] = normalize_age_series(df["idade"])

        for num_col in [
            "defasagem",
            "inde_22",
            "inde_23",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]:
            if num_col in df.columns:
                df[num_col] = coerce_numeric_series(df[num_col])
        return df

    if DATA_PATH.exists():
        return load_and_prepare_data(DATA_PATH)

    raise FileNotFoundError("Base consolidada e arquivo bruto indisponiveis para modelagem.")


## 3) Engenharia de target: risco de defasagem no ano subsequente

Definicao usada (ajustavel):

`risco_defasagem_t1 = 1 se (IAN_t+1 <= 5.0) E (IDA_t+1 <= 6.0)`

In [3]:
df = load_consolidated_or_raw().copy()

expected_cols = [
    "ra",
    "ano_referencia",
    "idade",
    "genero",
    "instituicao_ensino",
    "fase_programa",
    "turma",
    "pedra_ano",
    "inde_22",
    "inde_23",
    "inde_ano",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "defasagem",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
    "ativo_inativo",
]
for col in expected_cols:
    if col not in df.columns:
        df[col] = np.nan

base = df.rename(columns={"ano_referencia": "ano_base"}).copy()
future = df[["ra", "ano_referencia", "ian", "ida", "inde_ano"]].copy()
future = future.rename(
    columns={
        "ano_referencia": "ano_base",
        "ian": "ian_prox",
        "ida": "ida_prox",
        "inde_ano": "inde_prox",
    }
)
future["ano_base"] = future["ano_base"] - 1

model_df = base.merge(future, on=["ra", "ano_base"], how="left")
model_df = model_df.sort_values(["ra", "ano_base"]).reset_index(drop=True)

model_df["delta_ian_prox"] = model_df["ian_prox"] - model_df["ian"]
model_df["delta_ida_prox"] = model_df["ida_prox"] - model_df["ida"]
model_df["delta_inde_hist"] = model_df["inde_ano"] - model_df.groupby("ra")["inde_ano"].shift(1)
model_df["media_notas"] = model_df[["nota_matematica", "nota_portugues", "nota_ingles"]].mean(axis=1)
model_df["media_comportamental"] = model_df[["iaa", "ieg", "ips", "ipp"]].mean(axis=1)
model_df["desalinhamento_autoavaliacao"] = model_df["iaa"] - model_df[["ida", "ieg"]].mean(axis=1)

model_df["target_disponivel"] = model_df["ian_prox"].notna() & model_df["ida_prox"].notna()
model_df["risco_defasagem_t1"] = (
    (model_df["ian_prox"] <= TARGET_IAN_THRESHOLD)
    | (model_df["ida_prox"] <= TARGET_IDA_THRESHOLD)
    | (model_df["delta_ian_prox"] <= -1.0)
).astype("Int64")

labeled_df = model_df[(model_df["target_disponivel"]) & (model_df["ano_base"].isin([2022, 2023]))].copy()

print("Base total:", df.shape)
print("Base rotulada para modelagem:", labeled_df.shape)
display(labeled_df[["ano_base", "risco_defasagem_t1"]].value_counts().rename("contagem").reset_index())

class_balance = (
    labeled_df.groupby("ano_base", as_index=False)["risco_defasagem_t1"]
    .mean()
    .rename(columns={"risco_defasagem_t1": "taxa_risco"})
)
class_balance["taxa_risco"] = class_balance["taxa_risco"] * 100
display(class_balance.round(2))

fig_balance = px.bar(
    class_balance,
    x="ano_base",
    y="taxa_risco",
    text_auto=".1f",
    title="Taxa de risco no target por ano-base",
    labels={"ano_base": "Ano-base", "taxa_risco": "Taxa de risco (%)"},
)
fig_balance.show()


Base total: (3027, 35)
Base rotulada para modelagem: (1267, 39)


,ano_base,risco_defasagem_t1,contagem
0,2023,0,559
1,2022,0,434
2,2022,1,140
3,2023,1,134


,ano_base,taxa_risco
0,2022,24.39
1,2023,19.34


## 4) Split temporal, preprocessamento e tratamento de desbalanceamento

Estrategia temporal:
- treino: ano-base 2022 (prevendo 2023)
- teste: ano-base 2023 (prevendo 2024)

In [4]:
numeric_features = [
    "idade",
    "inde_22",
    "inde_23",
    "inde_ano",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "defasagem",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
    "media_notas",
    "media_comportamental",
    "desalinhamento_autoavaliacao",
    "delta_inde_hist",
]
categorical_features = ["genero", "instituicao_ensino", "fase_programa", "turma", "pedra_ano", "ativo_inativo"]
feature_columns = numeric_features + categorical_features

for col in feature_columns:
    if col not in labeled_df.columns:
        labeled_df[col] = np.nan

train_df = labeled_df[labeled_df["ano_base"] == 2022].copy()
test_df = labeled_df[labeled_df["ano_base"] == 2023].copy()
if train_df.empty or test_df.empty:
    raise ValueError("Split temporal invalido. Verifique os dados por ano-base.")

X_train = train_df[feature_columns].copy()
y_train = train_df["risco_defasagem_t1"].astype(int)
X_test = test_df[feature_columns].copy()
y_test = test_df["risco_defasagem_t1"].astype(int)

print("Treino:", X_train.shape, "| taxa positiva:", round(y_train.mean(), 3))
print("Teste:", X_test.shape, "| taxa positiva:", round(y_test.mean(), 3))

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", KNNImputer(n_neighbors=7, weights="distance")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)


Treino original: (574, 19) | taxa positiva: 0.244
Treino balanceado: (868, 19) | taxa positiva: 0.5
Teste: (693, 19) | taxa positiva: 0.193


## 5) Treino e comparacao de modelos

Modelos:
- Regressao Logistica (baseline interpretavel)
- Random Forest (maior capacidade preditiva)

A otimizacao prioriza Recall via ajuste de threshold.

In [5]:
models = {
    "Regressao Logistica": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=600,
        max_depth=10,
        min_samples_leaf=4,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=800,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}


def evaluate_threshold_grid(y_true: pd.Series, y_prob: np.ndarray, thresholds: np.ndarray) -> pd.DataFrame:
    rows = []
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        rows.append(
            {
                "threshold": float(threshold),
                "accuracy": accuracy_score(y_true, y_pred),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "f1": f1_score(y_true, y_pred, zero_division=0),
            }
        )
    return pd.DataFrame(rows)


def choose_threshold(threshold_df: pd.DataFrame, recall_floor: float) -> float:
    valid = threshold_df[threshold_df["recall"] >= recall_floor].copy()
    if valid.empty:
        valid = threshold_df.copy()
    winner = valid.sort_values(["f1", "precision", "recall", "accuracy"], ascending=False).iloc[0]
    return float(winner["threshold"])


def evaluate_probabilities(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "y_pred": y_pred,
        "report": classification_report(y_true, y_pred, output_dict=True, zero_division=0),
    }


cv_scoring = {
    "accuracy": "accuracy",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall": "recall",
    "f1": "f1",
}

results = {}
cv_rows = []

for model_name, estimator in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    cv_scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=cv_scoring,
        n_jobs=1,
        return_train_score=False,
    )

    cv_rows.append(
        {
            "modelo": model_name,
            "cv_accuracy_mean": cv_scores["test_accuracy"].mean(),
            "cv_accuracy_std": cv_scores["test_accuracy"].std(),
            "cv_roc_auc_mean": cv_scores["test_roc_auc"].mean(),
            "cv_roc_auc_std": cv_scores["test_roc_auc"].std(),
            "cv_pr_auc_mean": cv_scores["test_pr_auc"].mean(),
            "cv_pr_auc_std": cv_scores["test_pr_auc"].std(),
            "cv_recall_mean": cv_scores["test_recall"].mean(),
            "cv_f1_mean": cv_scores["test_f1"].mean(),
        }
    )

    oof_prob = cross_val_predict(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=1,
    )[:, 1]
    threshold_table = evaluate_threshold_grid(y_train, oof_prob, THRESHOLD_GRID)
    threshold = choose_threshold(threshold_table, recall_floor=RECALL_OBJECTIVE)

    pipeline.fit(X_train, y_train)
    test_prob = pipeline.predict_proba(X_test)[:, 1]
    metrics = evaluate_probabilities(y_test, test_prob, threshold)

    results[model_name] = {
        "pipeline": pipeline,
        "threshold": threshold,
        "metrics": metrics,
        "test_prob": test_prob,
        "oof_prob": oof_prob,
        "threshold_table": threshold_table,
    }

cv_metrics_table = pd.DataFrame(cv_rows).sort_values(["cv_pr_auc_mean", "cv_recall_mean"], ascending=False)
metrics_table = pd.DataFrame(
    [
        {
            "modelo": name,
            "threshold": result["threshold"],
            "accuracy": result["metrics"]["accuracy"],
            "precision": result["metrics"]["precision"],
            "recall": result["metrics"]["recall"],
            "f1": result["metrics"]["f1"],
            "roc_auc": result["metrics"]["roc_auc"],
            "pr_auc": result["metrics"]["pr_auc"],
            "brier": result["metrics"]["brier"],
        }
        for name, result in results.items()
    ]
).sort_values(["pr_auc", "recall", "roc_auc"], ascending=False)

print("Resumo de validacao cruzada (treino 2022):")
display(cv_metrics_table.round(4))

print("Resumo no teste temporal (2023):")
display(metrics_table.round(4))


,modelo,threshold,accuracy,precision,recall,f1,roc_auc
1,Random Forest,0.6731,0.8341,0.6727,0.2761,0.3915,0.8128
0,Regressao Logistica,0.5976,0.8211,0.6042,0.2164,0.3187,0.7826


## 6) Avaliacao: matriz de confusao e ROC-AUC

In [6]:
for model_name, result in results.items():
    y_pred = result["metrics"]["y_pred"]
    cm = confusion_matrix(y_test, y_pred)
    fig_cm = go.Figure(
        data=go.Heatmap(
            z=cm,
            x=["Pred 0", "Pred 1"],
            y=["Real 0", "Real 1"],
            text=cm,
            texttemplate="%{text}",
            colorscale="Blues",
        )
    )
    fig_cm.update_layout(title=f"Matriz de confusao - {model_name}")
    fig_cm.show()

roc_fig = go.Figure()
pr_fig = go.Figure()
for model_name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result["test_prob"])
    precision, recall, _ = precision_recall_curve(y_test, result["test_prob"])

    roc_fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{model_name} (AUC={result['metrics']['roc_auc']:.3f})",
        )
    )

    pr_fig.add_trace(
        go.Scatter(
            x=recall,
            y=precision,
            mode="lines",
            name=f"{model_name} (PR-AUC={result['metrics']['pr_auc']:.3f})",
        )
    )

roc_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatorio", line=dict(dash="dash")))
roc_fig.update_layout(
    title="Curva ROC - comparacao de modelos",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    xaxis=dict(constrain="domain"),
)
roc_fig.show()

pr_fig.update_layout(
    title="Curva Precision-Recall - comparacao de modelos",
    xaxis_title="Recall",
    yaxis_title="Precision",
)
pr_fig.show()

best_model_name_eval = metrics_table.iloc[0]["modelo"]
best_result_eval = results[best_model_name_eval]

threshold_curve_best = best_result_eval["threshold_table"].copy()
fig_threshold = px.line(
    threshold_curve_best,
    x="threshold",
    y=["precision", "recall", "f1"],
    markers=True,
    title=f"Curva de threshold (OOF treino) - {best_model_name_eval}",
    labels={"value": "Score", "threshold": "Threshold"},
)
fig_threshold.add_vline(
    x=best_result_eval["threshold"],
    line_dash="dash",
    line_color="red",
    annotation_text=f"Threshold selecionado: {best_result_eval['threshold']:.2f}",
)
fig_threshold.show()

calibration_df = pd.DataFrame({"prob": best_result_eval["test_prob"], "real": y_test.values})
calibration_df["faixa_prob"] = pd.cut(calibration_df["prob"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
calibration_table = (
    calibration_df.groupby("faixa_prob", as_index=False)
    .agg(amostra=("real", "size"), taxa_real=("real", "mean"), prob_media=("prob", "mean"))
)
calibration_table["taxa_real"] = calibration_table["taxa_real"] * 100
calibration_table["prob_media"] = calibration_table["prob_media"] * 100

print(f"Calibracao por faixas - {best_model_name_eval}")
display(calibration_table.round(2))


## 7) Explicabilidade: Feature Importance (e SHAP opcional)

In [7]:
best_model_name = metrics_table.iloc[0]["modelo"]
best_pipeline = results[best_model_name]["pipeline"]
best_threshold = results[best_model_name]["threshold"]

print("Melhor modelo (teste temporal):", best_model_name)
print("Threshold otimizado no treino (OOF):", round(best_threshold, 4))

fitted_preprocessor = best_pipeline.named_steps["preprocessor"]
fitted_model = best_pipeline.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(fitted_model, "feature_importances_"):
    importances = fitted_model.feature_importances_
    importance_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
else:
    coef = np.abs(fitted_model.coef_[0])
    importance_df = pd.DataFrame({"feature": feature_names, "importance": coef}).sort_values("importance", ascending=False)

display(importance_df.head(25))

fig_importance = px.bar(
    importance_df.head(20).sort_values("importance", ascending=True),
    x="importance",
    y="feature",
    orientation="h",
    title=f"Top 20 fatores mais relevantes - {best_model_name}",
    labels={"importance": "Importancia", "feature": "Feature"},
)
fig_importance.show()

report_df = pd.DataFrame(results[best_model_name]["metrics"]["report"]).T
print("Relatorio de classificacao (teste 2023):")
display(report_df.round(4))

try:
    import shap

    X_test_sample = X_test.sample(min(250, len(X_test)), random_state=RANDOM_STATE)
    X_test_transformed = fitted_preprocessor.transform(X_test_sample)
    if hasattr(X_test_transformed, "toarray"):
        X_test_transformed = X_test_transformed.toarray()

    if hasattr(fitted_model, "feature_importances_"):
        explainer = shap.TreeExplainer(fitted_model)
        shap_values = explainer.shap_values(X_test_transformed)
        if isinstance(shap_values, list):
            shap_matrix = shap_values[1]
        else:
            shap_matrix = shap_values
    else:
        explainer = shap.LinearExplainer(fitted_model, X_test_transformed)
        shap_matrix = explainer.shap_values(X_test_transformed)

    mean_abs_shap = np.abs(shap_matrix).mean(axis=0)
    shap_df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap}).sort_values("mean_abs_shap", ascending=False)
    display(shap_df.head(15))

    fig_shap = px.bar(
        shap_df.head(15).sort_values("mean_abs_shap", ascending=True),
        x="mean_abs_shap",
        y="feature",
        orientation="h",
        title="Top 15 features por |SHAP| medio",
        labels={"mean_abs_shap": "|SHAP| medio", "feature": "Feature"},
    )
    fig_shap.show()
except Exception as shap_error:
    print("SHAP nao executado (mantendo apenas feature importance):", shap_error)


Melhor modelo: Random Forest
Threshold otimizado: 0.6731


,feature,importance
1,num__inde_ano,0.177234
3,num__ida,0.112614
9,num__nota_matematica,0.105802
7,num__ipv,0.096706
0,num__idade,0.069371
4,num__ieg,0.062216
10,num__nota_portugues,0.060538
8,num__defasagem,0.044264
2,num__ian,0.033084
5,num__iaa,0.032289


SHAP nao executado (usando apenas Feature Importance): Per-column arrays must each be 1-dimensional


## 8) Exportacao de artefatos (modelo, scaler e score 2024)

In [8]:
default_numeric = {
    col: float(labeled_df[col].median()) if labeled_df[col].notna().any() else 0.0
    for col in numeric_features
}
default_categorical = {}
for col in categorical_features:
    mode_series = labeled_df[col].dropna().astype(str)
    default_categorical[col] = mode_series.mode().iloc[0] if not mode_series.empty else "Nao Informado"

category_options = {col: sorted(labeled_df[col].dropna().astype(str).unique().tolist()) for col in categorical_features}

best_metrics = results[best_model_name]["metrics"]
risk_bands = {
    "baixo_max": float(max(0.20, best_threshold - 0.15)),
    "alto_min": float(best_threshold),
}

training_info = {
    "train_years": [2022],
    "test_years": [2023],
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "target_positive_rate_train": float(y_train.mean()),
    "target_positive_rate_test": float(y_test.mean()),
    "best_model_test_metrics": {
        "accuracy": float(best_metrics["accuracy"]),
        "precision": float(best_metrics["precision"]),
        "recall": float(best_metrics["recall"]),
        "f1": float(best_metrics["f1"]),
        "roc_auc": float(best_metrics["roc_auc"]),
        "pr_auc": float(best_metrics["pr_auc"]),
        "brier": float(best_metrics["brier"]),
    },
}

bundle = {
    "model_name": best_model_name,
    "pipeline": best_pipeline,
    "threshold": float(best_threshold),
    "risk_bands": risk_bands,
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target_rule": {
        "description": "risco=1 se (IAN_t+1 <= 5.0) OU (IDA_t+1 <= 6.0) OU (queda_ian_t+1 <= -1.0)",
        "ian_threshold": TARGET_IAN_THRESHOLD,
        "ida_threshold": TARGET_IDA_THRESHOLD,
        "delta_ian_threshold": -1.0,
    },
    "healthy_thresholds": {"ian": 5.0, "ida": 6.0, "ieg": 7.0, "ips": 7.0, "ipp": 7.0, "ipv": 7.0},
    "default_inputs": {**default_numeric, **default_categorical},
    "category_options": category_options,
    "training_info": training_info,
    "model_comparison_cv": cv_metrics_table.to_dict(orient="records"),
    "model_comparison_test": metrics_table.to_dict(orient="records"),
    "threshold_curve_train_cv": results[best_model_name]["threshold_table"].to_dict(orient="records"),
}

model_path = ARTIFACT_DIR / "modelo_risco_defasagem.pkl"
with model_path.open("wb") as file:
    pickle.dump(bundle, file)

scaler_obj = best_pipeline.named_steps["preprocessor"].named_transformers_["num"].named_steps["scaler"]
scaler_path = ARTIFACT_DIR / "scaler_numerico.pkl"
with scaler_path.open("wb") as file:
    pickle.dump(scaler_obj, file)

importance_path = ARTIFACT_DIR / "feature_importance.csv"
importance_df.to_csv(importance_path, index=False)

model_compare_path = ARTIFACT_DIR / "model_comparison_test.csv"
metrics_table.to_csv(model_compare_path, index=False)

threshold_curve_path = ARTIFACT_DIR / "threshold_curve_best_model.csv"
results[best_model_name]["threshold_table"].to_csv(threshold_curve_path, index=False)

score_2024 = model_df[model_df["ano_base"] == 2024].copy()
if not score_2024.empty:
    X_2024 = score_2024[feature_columns].copy()
    score_2024["prob_risco"] = best_pipeline.predict_proba(X_2024)[:, 1]
    score_2024["risco_predito"] = (score_2024["prob_risco"] >= best_threshold).astype(int)
    score_2024["nivel_risco"] = pd.cut(
        score_2024["prob_risco"],
        bins=[-0.01, risk_bands["baixo_max"], risk_bands["alto_min"], 1.0],
        labels=["Baixo", "Moderado", "Alto"],
    )
    score_2024 = score_2024.sort_values("prob_risco", ascending=False)
    score_2024_out = ARTIFACT_DIR / "predicoes_risco_2024.csv"
    score_2024[["ra", "ano_base", "prob_risco", "risco_predito", "nivel_risco"]].to_csv(score_2024_out, index=False)

print("Artefatos exportados:")
print("-", model_path.resolve())
print("-", scaler_path.resolve())
print("-", importance_path.resolve())
print("-", model_compare_path.resolve())
print("-", threshold_curve_path.resolve())
if not score_2024.empty:
    print("-", score_2024_out.resolve())


Artefatos exportados:
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\modelo_risco_defasagem.pkl
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\scaler_numerico.pkl
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\feature_importance.csv
- C:\Users\vinic\OneDrive\Trabalhos\tech_desafio_fase5\artifacts\predicoes_risco_2024.csv
